<a href="https://colab.research.google.com/github/renatofb98/Data_science_projects/blob/main/extracao_dados_imagem.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
# ============================================================
# INSTALAÇÃO
# ============================================================

!apt-get -qq update
!apt-get -qq install -y tesseract-ocr tesseract-ocr-por tesseract-ocr-eng

!pip -q install pymupdf pytesseract pandas openpyxl tqdm pillow

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Selecting previously unselected package tesseract-ocr-por.
(Reading database ... 122809 files and directories currently installed.)
Preparing to unpack .../tesseract-ocr-por_1%3a4.1.0-2_all.deb ...
Unpacking tesseract-ocr-por (1:4.1.0-2) ...
Setting up tesseract-ocr-por (1:4.1.0-2) ...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.8/25.8 MB 73.2 MB/s eta 0:00:00


In [4]:
# ============================================================
# UPLOAD DO PDF
# ============================================================

from google.colab import files

uploaded = files.upload()

PDF_PATH = next(iter(uploaded.keys()))

print("PDF selecionado:", PDF_PATH)

Saving Manutenções 01092025 a 01092026.pdf to Manutenções 01092025 a 01092026.pdf
PDF selecionado: Manutenções 01092025 a 01092026.pdf


In [5]:
# ============================================================
# IMPORTAÇÕES
# ============================================================

import fitz
import pytesseract
import pandas as pd
import re
import unicodedata
import sqlite3
import os

from PIL import Image, ImageOps
from io import BytesIO
from tqdm.auto import tqdm


# ============================================================
# CONFIGURAÇÕES
# ============================================================

DPI = 250

# O OCR em português + inglês funcionou melhor para placas/modelos
OCR_LANG = "por+eng"
OCR_CONFIG = "--oem 3 --psm 6"


# ------------------------------------------------------------
# Limites horizontais das colunas.
#
# Foram definidos proporcionalmente à largura da página,
# portanto continuam funcionando mesmo alterando o DPI.
# ------------------------------------------------------------

COLUNAS_POSICAO = [
    ("data",               0.000, 0.075),
    ("km_ocorrencia",      0.075, 0.137),
    ("proximo_km",         0.137, 0.202),
    ("prestadora_servico", 0.202, 0.397),

    ("qtd_item",            0.397, 0.454),
    ("vl_unitario",         0.454, 0.520),
    ("vl_oleo",             0.520, 0.573),
    ("filt_oleo",           0.573, 0.623),
    ("filt_comb_a",         0.623, 0.677),
    ("filt_comb_b",         0.677, 0.739),
    ("filtro_ar",           0.739, 0.790),
    ("oleo_dif",            0.790, 0.845),
    ("vl_outros",           0.845, 0.906),
    ("total_ocorrencia",    0.906, 1.000),
]


# Setores encontrados/esperados no relatório
SETORES = [
    "FUNDO SOCIAL E SOLIDARIEDADE",
    "ADMINISTRAÇÃO / GABINETE",
    "ASSISTÊNCIA SOCIAL",
    "CONSELHO TUTELAR",
    "GUARDA MUNICIPAL",
    "MEIO AMBIENTE",
    "EDUCAÇÃO",
    "TRÂNSITO",
    "ESPORTES",
    "SAÚDE",
    "OBRAS",
]


# ============================================================
# FUNÇÕES AUXILIARES
# ============================================================

def normalizar_texto(texto):
    """
    Remove acentos, padroniza espaços e converte para maiúsculas.
    Usado apenas para comparação, sem alterar o texto original.
    """
    texto = str(texto)

    texto = unicodedata.normalize("NFKD", texto)
    texto = "".join(
        c for c in texto
        if not unicodedata.combining(c)
    )

    texto = re.sub(r"\s+", " ", texto).strip().upper()

    return texto


def normalizar_comparacao(texto):
    """
    Igual à normalização acima, mas também normaliza o uso da barra.
    Ex.:
    ADMINISTRAÇÃO / GABINETE
    ADMINISTRACAO/GABINETE

    são considerados equivalentes.
    """
    texto = normalizar_texto(texto)
    texto = re.sub(r"\s*/\s*", "/", texto)

    return texto


# ============================================================
# SEPARAÇÃO ENTRE ESPECIFICAÇÃO E SETOR
# ============================================================

def separar_setor(texto):
    """
    Exemplo:

    'CDV-1467 GM/ S10 2.4 RONTAN CAMIONETA 2002 OBRAS'

    retorna:

    especificação:
        CDV-1467 GM/ S10 2.4 RONTAN CAMIONETA 2002

    setor:
        OBRAS
    """

    tokens = texto.split()

    for setor in sorted(
        SETORES,
        key=lambda x: len(x),
        reverse=True
    ):

        setor_normalizado = normalizar_comparacao(setor)

        # Testa diferentes tamanhos de sufixo
        for quantidade in range(1, min(7, len(tokens)) + 1):

            sufixo = " ".join(tokens[-quantidade:])

            if normalizar_comparacao(sufixo) == setor_normalizado:

                especificacao = " ".join(
                    tokens[:-quantidade]
                ).strip()

                return especificacao, setor

    # Caso OCR não reconheça o setor
    return texto.strip(), None


# ============================================================
# IDENTIFICAÇÃO DA PLACA
# ============================================================

PADRAO_PLACA_ANTIGA = re.compile(
    r"^[A-Z]{3}-?\d{4}$"
)

PADRAO_PLACA_MERCOSUL = re.compile(
    r"^[A-Z]{3}\d[A-Z]\d{2}$"
)


def extrair_placa(especificacao):

    if not especificacao:
        return None

    primeiro_token = especificacao.split()[0].upper()

    primeiro_token = re.sub(
        r"[^A-Z0-9-]",
        "",
        primeiro_token
    )

    if (
        PADRAO_PLACA_ANTIGA.match(primeiro_token)
        or
        PADRAO_PLACA_MERCOSUL.match(primeiro_token)
    ):
        return primeiro_token

    return None


# ============================================================
# PARSE DO CABEÇALHO DO VEÍCULO
# ============================================================

def interpretar_veiculo(texto):
    """
    Cabeçalhos têm formato semelhante a:

    3 - CDV-1467 GM/ S10 2.4 RONTAN CAMIONETA 2002 OBRAS
    """

    match = re.match(
        r"^\s*(\d{1,4})\s*[-–—]\s*(.+?)\s*$",
        texto
    )

    if not match:
        return None

    codigo_bem = int(match.group(1))

    restante = match.group(2).strip()

    especificacao, setor = separar_setor(restante)

    placa = extrair_placa(especificacao)

    return {
        "codigo_bem": codigo_bem,
        "placa": placa,
        "especificacao_veiculo": especificacao,
        "setor": setor,
        "cabecalho_veiculo_ocr": texto,
    }


# ============================================================
# CONVERSÃO DE NÚMEROS BRASILEIROS
# ============================================================

def numero_br(valor):

    if valor is None:
        return None

    valor = str(valor).strip()

    if not valor:
        return None

    # Pequena correção de OCR
    valor = valor.replace("O", "0").replace("o", "0")

    valor = re.sub(
        r"[^0-9,.\-]",
        "",
        valor
    )

    if not valor:
        return None

    try:

        # 1.234,56
        if "," in valor:
            valor = valor.replace(".", "")
            valor = valor.replace(",", ".")
            return float(valor)

        # 120.000 = possível inteiro brasileiro
        if re.fullmatch(
            r"\d{1,3}(?:\.\d{3})+",
            valor
        ):
            return int(valor.replace(".", ""))

        # inteiro
        if re.fullmatch(r"-?\d+", valor):
            return int(valor)

        return float(valor)

    except:
        return None


# ============================================================
# CLASSIFICAÇÃO DAS LINHAS
# ============================================================

def linha_e_data(texto):

    return bool(
        re.match(
            r"^\s*\d{2}/\d{2}/\d{4}\b",
            texto
        )
    )


def linha_e_veiculo(texto):

    return bool(
        re.match(
            r"^\s*\d{1,4}\s*[-–—]\s*",
            texto
        )
    )


def linha_e_atividade(linha, largura_pagina):
    """
    Atividades como:

    REPARO
    REPARO PNEU
    VISTORIA
    REVISÃO TÉCNICA
    REPARAÇÃO E MANUTENÇÃO ...

    aparecem em caixa alta e alinhadas à esquerda.
    """

    texto = linha["text"].strip()

    if not texto:
        return False

    if linha_e_data(texto):
        return False

    if linha_e_veiculo(texto):
        return False

    texto_norm = normalizar_texto(texto)

    if texto_norm.startswith("TOTAL:"):
        return False

    if texto_norm.startswith("DESCRICAO OCORRENCIA"):
        return False

    # Deve começar próximo à margem esquerda
    if linha["left"] > largura_pagina * 0.10:
        return False

    # Evita pegar conteúdos muito largos
    if linha["right"] > largura_pagina * 0.65:
        return False

    letras = [
        c for c in texto
        if c.isalpha()
    ]

    if len(letras) < 4:
        return False

    proporcao_maiusculas = (
        sum(c.isupper() for c in letras)
        / len(letras)
    )

    return proporcao_maiusculas > 0.90


# ============================================================
# AGRUPAMENTO DAS PALAVRAS DO OCR EM LINHAS
# ============================================================

def agrupar_linhas_ocr(ocr):

    linhas = []

    agrupado = ocr.groupby(
        ["block_num", "par_num", "line_num"],
        sort=False
    )

    for chave, grupo in agrupado:

        grupo = grupo.sort_values("left")

        texto = " ".join(
            grupo["text"].astype(str)
        )

        linhas.append({
            "key": chave,
            "text": texto,

            "top": int(
                grupo["top"].min()
            ),

            "bottom": int(
                (
                    grupo["top"]
                    +
                    grupo["height"]
                ).max()
            ),

            "left": int(
                grupo["left"].min()
            ),

            "right": int(
                (
                    grupo["left"]
                    +
                    grupo["width"]
                ).max()
            ),

            "conf": float(
                grupo["conf"].mean()
            ),

            "words": grupo,
        })

    linhas = sorted(
        linhas,
        key=lambda x: (
            x["top"],
            x["left"]
        )
    )

    return linhas


# ============================================================
# DISTRIBUIÇÃO DAS PALAVRAS ENTRE AS COLUNAS
# ============================================================

def extrair_colunas_da_linha(
    palavras,
    largura_pagina
):

    celulas = {
        nome: ""
        for nome, _, _ in COLUNAS_POSICAO
    }

    palavras = palavras.sort_values("left")

    for _, palavra in palavras.iterrows():

        centro_x = (
            palavra["left"]
            +
            palavra["width"] / 2
        ) / largura_pagina

        for nome, inicio, fim in COLUNAS_POSICAO:

            if inicio <= centro_x < fim:

                celulas[nome] = (
                    celulas[nome]
                    + " "
                    + str(palavra["text"])
                ).strip()

                break

    return celulas


# ============================================================
# FUNÇÃO PRINCIPAL DE EXTRAÇÃO
# ============================================================

def extrair_pdf(
    pdf_path,
    dpi=250
):

    documento = fitz.open(pdf_path)

    registros = []

    # IMPORTANTE:
    # não zeramos essas variáveis a cada página,
    # porque alguns veículos continuam na página seguinte.
    veiculo_atual = None
    atividade_atual = None

    for numero_pagina in tqdm(
        range(len(documento)),
        desc="Processando páginas"
    ):

        pagina = documento[numero_pagina]

        # Renderiza PDF em imagem
        pix = pagina.get_pixmap(
            dpi=dpi,
            alpha=False
        )

        imagem = Image.open(
            BytesIO(
                pix.tobytes("png")
            )
        )

        # Melhora contraste
        imagem_ocr = ImageOps.grayscale(imagem)
        imagem_ocr = ImageOps.autocontrast(imagem_ocr)

        # OCR com coordenadas
        ocr = pytesseract.image_to_data(
            imagem_ocr,
            lang=OCR_LANG,
            config=OCR_CONFIG,
            output_type=pytesseract.Output.DATAFRAME
        )

        # Remove palavras vazias
        ocr = ocr.dropna(
            subset=["text"]
        ).copy()

        ocr["text"] = (
            ocr["text"]
            .astype(str)
            .str.strip()
        )

        ocr = ocr[
            ocr["text"] != ""
        ]

        linhas = agrupar_linhas_ocr(ocr)

        for linha in linhas:

            # Ignora cabeçalho institucional da página
            if linha["top"] < imagem.height * 0.18:
                continue

            texto = linha["text"].strip()

            # ------------------------------------------
            # NOVO VEÍCULO
            # ------------------------------------------

            if linha_e_veiculo(texto):

                veiculo_novo = interpretar_veiculo(
                    texto
                )

                if veiculo_novo:
                    veiculo_atual = veiculo_novo

                atividade_atual = None

                continue

            # ------------------------------------------
            # NOVA ATIVIDADE
            # ------------------------------------------

            if linha_e_atividade(
                linha,
                imagem.width
            ):

                atividade_atual = texto

                continue

            # ------------------------------------------
            # LINHA DE OCORRÊNCIA
            # ------------------------------------------

            if linha_e_data(texto):

                celulas = extrair_colunas_da_linha(
                    linha["words"],
                    imagem.width
                )

                registro = {
                    "pagina": numero_pagina + 1,

                    **(
                        veiculo_atual
                        if veiculo_atual
                        else {}
                    ),

                    "atividade": atividade_atual,

                    **celulas,

                    "confianca_ocr": round(
                        linha["conf"],
                        1
                    ),

                    "texto_linha_ocr": texto,
                }

                registros.append(registro)

    documento.close()

    return pd.DataFrame(registros)


# ============================================================
# EXECUÇÃO
# ============================================================

df = extrair_pdf(
    PDF_PATH,
    dpi=DPI
)

print()
print("Ocorrências extraídas:", len(df))

display(
    df.head(20)
)

Processando páginas:   0%|          | 0/39 [00:00<?, ?it/s]


Ocorrências extraídas: 844


,pagina,codigo_bem,placa,especificacao_veiculo,setor,cabecalho_veiculo_ocr,atividade,data,km_ocorrencia,proximo_km,...,vl_oleo,filt_oleo,filt_comb_a,filt_comb_b,filtro_ar,oleo_dif,vl_outros,total_ocorrencia,confianca_ocr,texto_linha_ocr
0,1,3,CDV-1467,CDV-1467 GM/ S10 2.4 RONTAN CAMIONETA 2002,OBRAS,3 - CDV-1467 GM/ S10 2.4 RONTAN CAMIONETA 2002...,REPARO PNEU,16/10/2025,,,...,"0,00","0,00","0,00",,,"0,00","24,00","24,00",95.7,"16/10/2025 BORRACHARIA E PNEUS FORMIGA 0 0,00 ..."
1,1,3,CDV-1467,CDV-1467 GM/ S10 2.4 RONTAN CAMIONETA 2002,OBRAS,3 - CDV-1467 GM/ S10 2.4 RONTAN CAMIONETA 2002...,REPARO,25/05/2026,,,...,"0,00","0,00","0,00",,,"0,00","1.130,00","1.130,00",94.2,"25/05/2026 MAICO FERNANDO BOLOGNESI 0 0,00 0,0..."
2,1,3,CDV-1467,CDV-1467 GM/ S10 2.4 RONTAN CAMIONETA 2002,OBRAS,3 - CDV-1467 GM/ S10 2.4 RONTAN CAMIONETA 2002...,REPARO,13/07/2026,,,...,"0,00","0,00","0,00",,,"0,00","750,00","750,00",95.8,"13/07/2026 JOAO BATISTA BUENO PINTO 0 0,00 0,0..."
3,1,3,CDV-1467,CDV-1467 GM/ S10 2.4 RONTAN CAMIONETA 2002,OBRAS,3 - CDV-1467 GM/ S10 2.4 RONTAN CAMIONETA 2002...,REPARO,22/07/2026,,,...,"0,00","0,00","0,00",,,"0,00","671,00","671,00",94.3,"22/07/2026 BORRACHARIA E PNEUS FORMIGA 0 0,00 ..."
4,1,3,CDV-1467,CDV-1467 GM/ S10 2.4 RONTAN CAMIONETA 2002,OBRAS,3 - CDV-1467 GM/ S10 2.4 RONTAN CAMIONETA 2002...,REPARO,27/07/2026,,,...,"0,00","0,00","0,00",,,"0,00","294,00","294,00",94.3,"27/07/2026 AUTO PECAS E MECANICA J.R.R. 0 0,00..."
5,1,6,DBA8C21,DBA8C21 FIAT/ FIORINO IE 2005,OBRAS,6 - DBA8C21 FIAT/ FIORINO IE 2005 OBRAS,REPARO,28/10/2025,,,...,"0,00","0,00","0,00",,,"0,00","160,00","160,00",95.1,"28/10/2025 RIMENIS BALDACIN OLIVEIRA 0 0,00 0,..."
6,1,6,DBA8C21,DBA8C21 FIAT/ FIORINO IE 2005,OBRAS,6 - DBA8C21 FIAT/ FIORINO IE 2005 OBRAS,REPARO PNEU,10/12/2025,,,...,"0,00","0,00","0,00",,,"0,00","149,00","149,00",95.9,"10/12/2025 BORRACHARIA E PNEUS FORMIGA 0 0,00 ..."
7,1,6,DBA8C21,DBA8C21 FIAT/ FIORINO IE 2005,OBRAS,6 - DBA8C21 FIAT/ FIORINO IE 2005 OBRAS,REPARO,29/04/2026,,,...,"0,00","0,00","0,00",,,"0,00","27,00","27,00",94.5,"29/04/2026 BORRACHARIA E PNEUS FORMIGA 0 0,00 ..."
8,1,6,DBA8C21,DBA8C21 FIAT/ FIORINO IE 2005,OBRAS,6 - DBA8C21 FIAT/ FIORINO IE 2005 OBRAS,REPARO,25/05/2026,,,...,"0,00","0,00","0,00",,,"0,00","260,00","260,00",94.3,"25/05/2026 UNITEC ABC MANUTENCAO VEIC 0 0,00 0..."
9,1,6,DBA8C21,DBA8C21 FIAT/ FIORINO IE 2005,OBRAS,6 - DBA8C21 FIAT/ FIORINO IE 2005 OBRAS,REPARO,16/06/2026,,,...,"0,00","0,00","0,00",,,"0,00","570,00","570,00",95.1,"16/06/2026 SCUDELER PNEUS E CENTRO 0 0,00 0,00..."


In [6]:
# ============================================================
# CONVERSÃO PARA TIPOS DE BANCO DE DADOS
# ============================================================

COLUNAS_NUMERICAS = [
    "km_ocorrencia",
    "proximo_km",
    "qtd_item",
    "vl_unitario",
    "vl_oleo",
    "filt_oleo",
    "filt_comb_a",
    "filt_comb_b",
    "filtro_ar",
    "oleo_dif",
    "vl_outros",
    "total_ocorrencia",
]


for coluna in COLUNAS_NUMERICAS:

    if coluna in df.columns:

        df[coluna] = df[coluna].apply(
            numero_br
        )


# Data real
df["data"] = pd.to_datetime(
    df["data"],
    format="%d/%m/%Y",
    errors="coerce"
)


# ------------------------------------------------------------
# Flag para revisão manual
# ------------------------------------------------------------

df["revisar_ocr"] = (
    (df["confianca_ocr"] < 80)
    |
    (df["codigo_bem"].isna())
    |
    (df["atividade"].isna())
    |
    (df["prestadora_servico"].fillna("").str.strip() == "")
    |
    (df["total_ocorrencia"].isna())
)


# Reorganização
ORDEM_COLUNAS = [
    "pagina",

    "codigo_bem",
    "placa",
    "especificacao_veiculo",
    "setor",

    "atividade",

    "data",
    "km_ocorrencia",
    "proximo_km",
    "prestadora_servico",

    "qtd_item",
    "vl_unitario",
    "vl_oleo",
    "filt_oleo",
    "filt_comb_a",
    "filt_comb_b",
    "filtro_ar",
    "oleo_dif",
    "vl_outros",
    "total_ocorrencia",

    "confianca_ocr",
    "revisar_ocr",

    "cabecalho_veiculo_ocr",
    "texto_linha_ocr",
]

ORDEM_COLUNAS = [
    c for c in ORDEM_COLUNAS
    if c in df.columns
]

df = df[ORDEM_COLUNAS]


print("Total de registros:", len(df))

print(
    "Registros marcados para revisão:",
    df["revisar_ocr"].sum()
)

display(df.head(20))

Total de registros: 844
Registros marcados para revisão: 0


,pagina,codigo_bem,placa,especificacao_veiculo,setor,atividade,data,km_ocorrencia,proximo_km,prestadora_servico,...,filt_comb_a,filt_comb_b,filtro_ar,oleo_dif,vl_outros,total_ocorrencia,confianca_ocr,revisar_ocr,cabecalho_veiculo_ocr,texto_linha_ocr
0,1,3,CDV-1467,CDV-1467 GM/ S10 2.4 RONTAN CAMIONETA 2002,OBRAS,REPARO PNEU,2025-10-16,NaN,None,BORRACHARIA E PNEUS FORMIGA,...,0.0,None,None,0.0,24.0,24.0,95.7,False,3 - CDV-1467 GM/ S10 2.4 RONTAN CAMIONETA 2002...,"16/10/2025 BORRACHARIA E PNEUS FORMIGA 0 0,00 ..."
1,1,3,CDV-1467,CDV-1467 GM/ S10 2.4 RONTAN CAMIONETA 2002,OBRAS,REPARO,2026-05-25,NaN,None,MAICO FERNANDO BOLOGNESI,...,0.0,None,None,0.0,1130.0,1130.0,94.2,False,3 - CDV-1467 GM/ S10 2.4 RONTAN CAMIONETA 2002...,"25/05/2026 MAICO FERNANDO BOLOGNESI 0 0,00 0,0..."
2,1,3,CDV-1467,CDV-1467 GM/ S10 2.4 RONTAN CAMIONETA 2002,OBRAS,REPARO,2026-07-13,NaN,None,JOAO BATISTA BUENO PINTO,...,0.0,None,None,0.0,750.0,750.0,95.8,False,3 - CDV-1467 GM/ S10 2.4 RONTAN CAMIONETA 2002...,"13/07/2026 JOAO BATISTA BUENO PINTO 0 0,00 0,0..."
3,1,3,CDV-1467,CDV-1467 GM/ S10 2.4 RONTAN CAMIONETA 2002,OBRAS,REPARO,2026-07-22,NaN,None,BORRACHARIA E PNEUS FORMIGA,...,0.0,None,None,0.0,671.0,671.0,94.3,False,3 - CDV-1467 GM/ S10 2.4 RONTAN CAMIONETA 2002...,"22/07/2026 BORRACHARIA E PNEUS FORMIGA 0 0,00 ..."
4,1,3,CDV-1467,CDV-1467 GM/ S10 2.4 RONTAN CAMIONETA 2002,OBRAS,REPARO,2026-07-27,NaN,None,AUTO PECAS E MECANICA J.R.R.,...,0.0,None,None,0.0,294.0,294.0,94.3,False,3 - CDV-1467 GM/ S10 2.4 RONTAN CAMIONETA 2002...,"27/07/2026 AUTO PECAS E MECANICA J.R.R. 0 0,00..."
5,1,6,DBA8C21,DBA8C21 FIAT/ FIORINO IE 2005,OBRAS,REPARO,2025-10-28,NaN,None,RIMENIS BALDACIN OLIVEIRA,...,0.0,None,None,0.0,160.0,160.0,95.1,False,6 - DBA8C21 FIAT/ FIORINO IE 2005 OBRAS,"28/10/2025 RIMENIS BALDACIN OLIVEIRA 0 0,00 0,..."
6,1,6,DBA8C21,DBA8C21 FIAT/ FIORINO IE 2005,OBRAS,REPARO PNEU,2025-12-10,NaN,None,BORRACHARIA E PNEUS FORMIGA,...,0.0,None,None,0.0,149.0,149.0,95.9,False,6 - DBA8C21 FIAT/ FIORINO IE 2005 OBRAS,"10/12/2025 BORRACHARIA E PNEUS FORMIGA 0 0,00 ..."
7,1,6,DBA8C21,DBA8C21 FIAT/ FIORINO IE 2005,OBRAS,REPARO,2026-04-29,NaN,None,BORRACHARIA E PNEUS FORMIGA,...,0.0,None,None,0.0,27.0,27.0,94.5,False,6 - DBA8C21 FIAT/ FIORINO IE 2005 OBRAS,"29/04/2026 BORRACHARIA E PNEUS FORMIGA 0 0,00 ..."
8,1,6,DBA8C21,DBA8C21 FIAT/ FIORINO IE 2005,OBRAS,REPARO,2026-05-25,NaN,None,UNITEC ABC MANUTENCAO VEIC,...,0.0,None,None,0.0,260.0,260.0,94.3,False,6 - DBA8C21 FIAT/ FIORINO IE 2005 OBRAS,"25/05/2026 UNITEC ABC MANUTENCAO VEIC 0 0,00 0..."
9,1,6,DBA8C21,DBA8C21 FIAT/ FIORINO IE 2005,OBRAS,REPARO,2026-06-16,NaN,None,SCUDELER PNEUS E CENTRO,...,0.0,None,None,0.0,570.0,570.0,95.1,False,6 - DBA8C21 FIAT/ FIORINO IE 2005 OBRAS,"16/06/2026 SCUDELER PNEUS E CENTRO 0 0,00 0,00..."


In [7]:
# ============================================================
# TABELA DE VEÍCULOS
# ============================================================

veiculos = (
    df[
        [
            "codigo_bem",
            "placa",
            "especificacao_veiculo",
            "setor",
            "cabecalho_veiculo_ocr"
        ]
    ]
    .drop_duplicates(
        subset=["codigo_bem"]
    )
    .sort_values("codigo_bem")
    .reset_index(drop=True)
)


# ============================================================
# ARQUIVOS DE SAÍDA
# ============================================================

CSV_OUT = "manutencoes_extraidas.csv"
XLSX_OUT = "manutencoes_extraidas.xlsx"
DB_OUT = "manutencoes.db"


# ------------------------------------------------------------
# CSV
# ------------------------------------------------------------

df.to_csv(
    CSV_OUT,
    index=False,
    sep=";",
    decimal=",",
    encoding="utf-8-sig"
)


# ------------------------------------------------------------
# EXCEL
# ------------------------------------------------------------

with pd.ExcelWriter(
    XLSX_OUT,
    engine="openpyxl"
) as writer:

    df.to_excel(
        writer,
        sheet_name="manutencoes",
        index=False
    )

    veiculos.to_excel(
        writer,
        sheet_name="veiculos",
        index=False
    )

    df[
        df["revisar_ocr"]
    ].to_excel(
        writer,
        sheet_name="revisar_ocr",
        index=False
    )


# ------------------------------------------------------------
# SQLITE
# ------------------------------------------------------------

conexao = sqlite3.connect(DB_OUT)

df.to_sql(
    "manutencoes",
    conexao,
    if_exists="replace",
    index=False
)

veiculos.to_sql(
    "veiculos",
    conexao,
    if_exists="replace",
    index=False
)

conexao.close()


print("Arquivos criados:")
print(" -", CSV_OUT)
print(" -", XLSX_OUT)
print(" -", DB_OUT)

Arquivos criados:
 - manutencoes_extraidas.csv
 - manutencoes_extraidas.xlsx
 - manutencoes.db


In [8]:
# Colunas mais importantes para uma conferência rápida

display(
    df[
        [
            "codigo_bem",
            "placa",
            "especificacao_veiculo",
            "setor",
            "atividade",
            "data",
            "prestadora_servico",
            "vl_outros",
            "total_ocorrencia",
            "revisar_ocr"
        ]
    ].head(50)
)

,codigo_bem,placa,especificacao_veiculo,setor,atividade,data,prestadora_servico,vl_outros,total_ocorrencia,revisar_ocr
0,3,CDV-1467,CDV-1467 GM/ S10 2.4 RONTAN CAMIONETA 2002,OBRAS,REPARO PNEU,2025-10-16,BORRACHARIA E PNEUS FORMIGA,24.0,24.0,False
1,3,CDV-1467,CDV-1467 GM/ S10 2.4 RONTAN CAMIONETA 2002,OBRAS,REPARO,2026-05-25,MAICO FERNANDO BOLOGNESI,1130.0,1130.0,False
2,3,CDV-1467,CDV-1467 GM/ S10 2.4 RONTAN CAMIONETA 2002,OBRAS,REPARO,2026-07-13,JOAO BATISTA BUENO PINTO,750.0,750.0,False
3,3,CDV-1467,CDV-1467 GM/ S10 2.4 RONTAN CAMIONETA 2002,OBRAS,REPARO,2026-07-22,BORRACHARIA E PNEUS FORMIGA,671.0,671.0,False
4,3,CDV-1467,CDV-1467 GM/ S10 2.4 RONTAN CAMIONETA 2002,OBRAS,REPARO,2026-07-27,AUTO PECAS E MECANICA J.R.R.,294.0,294.0,False
5,6,DBA8C21,DBA8C21 FIAT/ FIORINO IE 2005,OBRAS,REPARO,2025-10-28,RIMENIS BALDACIN OLIVEIRA,160.0,160.0,False
6,6,DBA8C21,DBA8C21 FIAT/ FIORINO IE 2005,OBRAS,REPARO PNEU,2025-12-10,BORRACHARIA E PNEUS FORMIGA,149.0,149.0,False
7,6,DBA8C21,DBA8C21 FIAT/ FIORINO IE 2005,OBRAS,REPARO,2026-04-29,BORRACHARIA E PNEUS FORMIGA,27.0,27.0,False
8,6,DBA8C21,DBA8C21 FIAT/ FIORINO IE 2005,OBRAS,REPARO,2026-05-25,UNITEC ABC MANUTENCAO VEIC,260.0,260.0,False
9,6,DBA8C21,DBA8C21 FIAT/ FIORINO IE 2005,OBRAS,REPARO,2026-06-16,SCUDELER PNEUS E CENTRO,570.0,570.0,False


In [9]:
from google.colab import files

files.download("manutencoes_extraidas.xlsx")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>